<div align="right"><a href="S30-PSD-Estimation-using-the-Periodogram.ipynb">Sprache wechseln zu Englisch 🇬🇧</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

### Traditionelle Spektrumschätzung mit dem Periodogramm

Das **Periodogramm** ist eine einfache Methode, um die Leistungsdichtespektren
$S_{XX}(\mathrm{e}^{j\Omega})$ auf Basis von $N$ Stichproben eines stationären Prozesses $X(k)$ zu schätzen.

$$
\mathrm{Per}_N^X(\Omega) := \frac{1}{N} \left| X(e^{j\Omega}) \right|^2,
\quad \text{mit} \quad X(e^{j\Omega}) = \mathcal{F}{ X(k) }, \quad k=0,\dots,N-1
$$

1. Erstelle zwei Modellprozesse $X_1(k), X_2(k)$ mit der Länge $N=2^{12}$ durch lineares Filtern eines weißen Gaußschen Rauschprozesses $N(k)$ mit Mittelwert null und Leistung eins, unter Verwendung der Filter $H_1(z)$ und $H_2(z)$:

$$
H_1(z) = 1 - 4 z^{-1} + 6 z^{-2} - 4 z^{-3} + z^{-4}, \quad
H_2(z) = \frac{1}{1 + 0.8, z^{-1}}.
$$

2. Plotte das „wahre“ Leistungsdichtespektrum $S_{X_i X_i}(\mathrm{e}^{j\Omega})$ für beide Filter.

3. Berechne die Periodogramme $\mathrm{Per}_N^{X_i}(\Omega)$ für $N = 2^6, 2^8, 2^{10}$ und $N = 2^{12}$ Stichproben für beide Prozesse.
   Warum nimmt die Varianz der Schätzung nicht ab, wenn die Anzahl der Stichproben $N$ steigt?


In [ ]:
# BEGIN SOLUTION
% Teilaufg. a:  Lineare Filterung
N     = 2^12;          % mittelwertfreies, gaussvert., weisses Rauschen
N_vor = 500;           % Anzahl der Abtastwerte um das System in den eingeschw.
                       % Zustand zu setzen, Leistung des Prozesses ist eins
n  = randn(1,N+N_vor);

% Koeffizientenvektor f�r die Filter H1 und H2, Filterung mit N
h1 = [1 -4 6 -4 +1]; 
x1_all = filter(h1,1,n); 
x1 = x1_all(N_vor+1:length(x1_all));  

h2 = [1 0.8];        
x2_all = filter(1,h2,n); 
x2=x2_all(N_vor+1:length(x2_all));

% Teilaufg. b:  "Wahres" Leistungsdichtespektrum fuer die beiden Filter
NFFT = 2^12;                              % FFT L�nge              

Sx1x1 = abs(fft(h1,NFFT)).^2;             % Berechnung des LDS
axis_Sx1=[0 1 0 2*max(Sx1x1)];            % Skalierung der Achse f�r die 
                                          % grafische Darst. in Teilaufg. c

Sx2x2 = ones(1,NFFT)./abs(fft(h2,NFFT)).^2;
axis_Sx2=[0 1 0 1.5*max(Sx2x2)];

fT = 0:1/NFFT:1-1/NFFT;                   % Skalierung der Achse f�r graf. Darst.

% # grafische Darstellung # %
figure;
plot(fT,Sx1x1);
axis([0 1 0 1.1*max(Sx1x1)]);
xlabel('\Omega/2\pi = f \cdot T');  
ylabel('S_{XX}(exp(j\cdot\Omega))');
title('Wahres Leistungsdichtespektrum S_{X1X1} von x_1');

figure;
plot(fT,Sx2x2); 
axis([0 1 0 1.1*max(Sx2x2)]);
xlabel('\Omega/2\pi = f \cdot T');  
ylabel('S_{XX}(exp(j\cdot\Omega))');
title('Wahres Leistungsdichtespektrum S_{X2X2} von x_2');

input('....Fuer naechste Teilaufgabe: RETURN druecken');

% Teilaufg. c:  Periodogramme und wahre Leistungsdichtespektren zeichnen 
fT = 0:1/NFFT:1-1/NFFT;

for k=[2^6, 2^8, 2^10, 2^12]
    Sxx=lper(x1(1:k) ,NFFT);

    % # grafische Darstellung # %
    figure;
    plot(fT,Sxx);
    hold on;
    plot(fT,Sx1x1,'--');
    xlabel('\Omega/2\pi = f \cdot T');  
    ylabel('Per(Om)');
    title(sprintf('Periodogramm von S_{X1X1}, N=%d', k));
    axis(axis_Sx1);
end

input('....Zum Fortfahren: RETURN druecken');

for k=[2^6, 2^8, 2^10, 2^12]
    Sxx=lper(x2(1:k) ,NFFT);

    % # grafische Darstellung # %
    figure;
    plot(fT,Sxx);
    hold on;
    plot(fT,Sx2x2,'--');
    xlabel('\Omega/2\pi = f \cdot T');
    ylabel('Per(Om)');
    title(sprintf('Periodogramm von S_{X2X2}, N=%d', k)); 
    axis(axis_Sx2);
end;
# END SOLUTION

### Lösung: Traditionelle Spektrumschätzung mit dem Periodogramm

a) Mit einem gegebenen Eingangssignalvektor `n = [n_0, ..., n_{N-1}]` berechnet die MATLAB-Funktion

```matlab
xi = filter(bi, ai, n)
```

das Ausgangssignal

```
xi = [x_{0,i}, ..., x_{N-1,i}]
```

des ARMA-Modells

$$
H_i(z) = \frac{B_i(z)}{A_i(z)} = \frac{\sum_{\mu=0}^{m} b_{\mu,i} z^{-\mu}}{\sum_{\nu=0}^{n} a_{\nu,i} z^{-\nu}},
$$

wobei die MA-Koeffizienten $b_{\mu,i}$ und die AR-Koeffizienten durch die Vektoren `bi` und `ai` gegeben sind.

Für $H_1(z)$: `b1 = [1, -4, 6, -4, 1]` und `a1 = 1`
Für $H_2(z)$: `b2 = 1` und `a2 = [1, 0.8]`

---

b) Für stationäres weißes Rauschen mit Varianz $\sigma_N^2 = 1$ gelten für die wahren Leistungsdichtespektren des Ausgangssignals $x_i(k)$:

$$
S_{X_i X_i}(e^{j\Omega}) = \sigma_N^2 \cdot |H_i(e^{j\Omega})|^2 = \left| \frac{B_i(e^{j\Omega})}{A_i(e^{j\Omega})} \right|^2 = \frac{| \mathcal{F}{b_i(k)} |^2}{| \mathcal{F}{a_i(k)} |^2}.
$$

Um die wahren Leistungsdichtespektren $S_{X_1 X_1}(e^{j\Omega})$ und $S_{X_2 X_2}(e^{j\Omega})$ darzustellen, wird die diskrete Fourier-Transformation $\mathcal{F}{\cdot}$ durch eine $2^{12}$-Punkt-FFT approximiert.

<img src="./../pngs/ldsv8a02_sx1x1.png" width="250"> <img src="./../pngs/ldsv8a02_sx2x2.png" width="250">

---

c) Die folgenden Abbildungen zeigen die Periodogramme $\mathrm{Per}_N^{X_i}(\Omega)$ für $N = 2^6 = 64$, $2^8 = 256$, $2^{10} = 1028$ und $N = 2^{12} = 4096$ Datenwerte jedes Stichprobenprozesses $X_1(k)$ und $X_2(k)$.

<img src="./../pngs/ldsv8a02_perx1N64.png" width="250"> <img src="./../pngs/ldsv8a02_perx1N256.png" width="250"> <img src="./../pngs/ldsv8a02_perx1N1024.png" width="250"> <img src="./../pngs/ldsv8a02_perx1N4096.png" width="250">

<img src="./../pngs/ldsv8a02_perx2N64.png" width="250"> <img src="./../pngs/ldsv8a02_perx2N256.png" width="250"> <img src="./../pngs/ldsv8a02_perx2N1024.png" width="250"> <img src="./../pngs/ldsv8a02_perx2N4096.png" width="250">

Die diskrete Fourier-Transformation (DTFT) $\mathcal{F}{\cdot}$ wird erneut durch eine $2^{12}$-Punkte-FFT approximiert. Das wahre Leistungsdichtespektrum $S_{X_i X_i}(e^{j\Omega})$ ist durch eine gestrichelte Linie dargestellt.
Es ist erkennbar, dass die Spektrumschätzung mittels Periodogramm unbefriedigende Ergebnisse liefert, bedingt durch die Varianz

$$
\mathrm{Var}{ \mathrm{Per}*N^{X_i}(\Omega) } \sim S*{X_i X_i}(e^{j\Omega})^2.
$$

Nur in den Frequenzintervallen, in denen das wahre Leistungsdichtespektrum $S_{X_i X_i}(e^{j\Omega})$ verschwindet, liefert das Periodogramm den Wert null (mit verschwindender Varianz). In Frequenzbereichen mit Spektralanteilen erlaubt das Periodogramm keine Vorhersage von $S_{X_i X_i}(e^{j\Omega})$, da die Schätzung durch eine zu hohe Varianz entwertet wird. Eine Erhöhung von $N$ ändert dieses Verhalten ebenfalls nicht.
Das Bias-Verhalten verbessert sich, wenn $N$ erhöht wird.

**Hinweis:** Da die Stichprobenfunktion $x_i(k)$ des Rauschprozesses $X_i(k)$ zufällig ist, ergeben sich andere Periodogramme beim Aufruf der MATLAB-Dateien `l12.m`. Die Leistungsdichte muss wiederholt geschätzt werden, um Informationen über Bias oder Varianz des Periodogramms zu erhalten. Anschließend können Mittelwerttrends und Mittelwert ± Standardabweichungs-Intervalle geplottet werden.